In [2]:
from sqlalchemy import create_engine, inspect
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np
import getpass

# 비밀번호 입력
password = getpass.getpass("MySQL 비밀번호: ")

# 새 DB
DB_NAME = "fintech_cohort_25_39"

# 연결 URL 생성
url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    database=DB_NAME
)

# 엔진 생성
engine = create_engine(url)

# 연결 확인
inspector = inspect(engine)
tables = inspector.get_table_names()

print("DB:", DB_NAME)
print("전체 테이블 수:", len(tables))

print("\n테이블 목록:")
for table in tables:
    print("-", table)

MySQL 비밀번호:  ········


DB: fintech_cohort_25_39
전체 테이블 수: 25

테이블 목록:
- _cohort
- accounts
- card_billing_statements
- card_payments
- cards
- customer_credit_behavior
- customer_debt_summary
- customer_employment_events
- customer_financial_profiles
- customer_logins
- customer_monthly_cashflow
- customer_real_estate
- customer_stock_lots
- customer_stock_positions
- customers
- devices
- fraud_cases
- household_children
- investment_accounts
- loan_repayments
- loans
- mydata_consents
- real_assets
- self_employed_businesses
- transactions


In [3]:
# ============================================================
# 1. 새 DB 기본 구조 확인
# ============================================================

# 고객 기본 정보
customers = pd.read_sql("""
SELECT *
FROM customers
""", engine)

# 금융 프로필
financial_profiles = pd.read_sql("""
SELECT
    customer_id,
    is_stock_investor,
    disposable_annual_income
FROM customer_financial_profiles
""", engine)

# 월별 현금흐름
cashflow_range = pd.read_sql("""
SELECT
    MIN(month) AS min_month,
    MAX(month) AS max_month,
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS customer_count
FROM customer_monthly_cashflow
""", engine)

print("=" * 65)
print("NEW DB BASIC CHECK")
print("=" * 65)

print("\n[1. customers]")
print("고객 수:", customers["customer_id"].nunique())

if "age" in customers.columns:
    print("연령 범위:", customers["age"].min(), "~", customers["age"].max())
    print("\n연령별 고객 수:")
    print(customers["age"].value_counts().sort_index())

print("\n[2. customer_financial_profiles]")
print("고객 수:", financial_profiles["customer_id"].nunique())
print("투자자 수:", (financial_profiles["is_stock_investor"] == 1).sum())
print("비투자자 수:", (financial_profiles["is_stock_investor"] == 0).sum())

print("\n[3. customer_monthly_cashflow]")
display(cashflow_range)

NEW DB BASIC CHECK

[1. customers]
고객 수: 13852

[2. customer_financial_profiles]
고객 수: 13852
투자자 수: 4079
비투자자 수: 9773

[3. customer_monthly_cashflow]


,min_month,max_month,row_count,customer_count
0,2024-11,2026-10,332448,13852


In [4]:
# ============================================================
# 2. cohort 및 주요 테이블 컬럼 확인
# ============================================================

tables_to_check = [
    "_cohort",
    "customers",
    "customer_financial_profiles",
    "customer_monthly_cashflow",
    "customer_stock_positions"
]

for table in tables_to_check:
    print("\n" + "=" * 65)
    print(table)
    print("=" * 65)
    
    columns = inspector.get_columns(table)
    
    for col in columns:
        print(f"{col['name']:<35} {col['type']}")


_cohort
customer_id                         BIGINT

customers
customer_id                         BIGINT
name                                TEXT COLLATE "utf8mb4_unicode_ci"
birth_date                          DATE
gender                              TEXT COLLATE "utf8mb4_unicode_ci"
region                              TEXT COLLATE "utf8mb4_unicode_ci"
city                                TEXT COLLATE "utf8mb4_unicode_ci"
income_band                         TEXT COLLATE "utf8mb4_unicode_ci"
occupation                          TEXT COLLATE "utf8mb4_unicode_ci"
credit_score                        BIGINT
kyc_level                           TEXT COLLATE "utf8mb4_unicode_ci"
risk_segment                        TEXT COLLATE "utf8mb4_unicode_ci"
signup_channel                      TEXT COLLATE "utf8mb4_unicode_ci"
created_at                          DATETIME
updated_at                          DATETIME

customer_financial_profiles
customer_id                         BIGINT
name              

In [5]:
# ============================================================
# 3. 코호트 검증
# ============================================================

cohort_check = pd.read_sql("""
SELECT
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS customer_count,
    SUM(is_stock_investor = 1) AS investor_count,
    SUM(is_stock_investor = 0) AS non_investor_count
FROM customer_financial_profiles
""", engine)

cohort_table = pd.read_sql("""
SELECT customer_id
FROM _cohort
""", engine)

profile_ids = pd.read_sql("""
SELECT customer_id
FROM customer_financial_profiles
""", engine)

print("=" * 65)
print("COHORT VALIDATION")
print("=" * 65)

display(cohort_check)

print("\n[1. _cohort]")
print("행 수:", len(cohort_table))
print("고객 수:", cohort_table["customer_id"].nunique())
print("중복 ID:", cohort_table["customer_id"].duplicated().sum())

print("\n[2. financial_profiles]")
print("행 수:", len(profile_ids))
print("고객 수:", profile_ids["customer_id"].nunique())
print("중복 ID:", profile_ids["customer_id"].duplicated().sum())

cohort_ids = set(cohort_table["customer_id"])
fp_ids = set(profile_ids["customer_id"])

print("\n[3. ID 일치 여부]")
print("_cohort에만 있는 고객:", len(cohort_ids - fp_ids))
print("financial_profiles에만 있는 고객:", len(fp_ids - cohort_ids))

print("\n[4. 연령별 고객 수]")
age_dist = pd.read_sql("""
SELECT
    age,
    COUNT(*) AS customer_count,
    SUM(is_stock_investor = 1) AS investor_count
FROM customer_financial_profiles
GROUP BY age
ORDER BY age
""", engine)

display(age_dist)

COHORT VALIDATION


,min_age,max_age,row_count,customer_count,investor_count,non_investor_count
0,24,38,13852,13852,4079.0,9773.0



[1. _cohort]
행 수: 13852
고객 수: 13852
중복 ID: 0

[2. financial_profiles]
행 수: 13852
고객 수: 13852
중복 ID: 0

[3. ID 일치 여부]
_cohort에만 있는 고객: 0
financial_profiles에만 있는 고객: 0

[4. 연령별 고객 수]


,age,customer_count,investor_count
0,24,1496,335.0
1,25,1334,302.0
2,26,1342,310.0
3,27,1337,284.0
4,28,1400,301.0
5,29,1276,271.0
6,30,606,236.0
7,31,636,257.0
8,32,593,229.0
9,33,672,269.0


In [6]:
# ============================================================
# 4. 투자 적극성 계산
#    - 기준기간: 2025-10 ~ 2026-09
#    - 대상: 주식 투자자(is_stock_investor = 1)
# ============================================================

# 1) 투자자 기본 정보
investor_base = pd.read_sql("""
SELECT
    customer_id,
    disposable_annual_income
FROM customer_financial_profiles
WHERE is_stock_investor = 1
""", engine)


# 2) 최근 12개월 투자이체 총액
investment_12m = pd.read_sql("""
SELECT
    customer_id,
    SUM(investment_transfer) AS investment_12m
FROM customer_monthly_cashflow
WHERE month BETWEEN '2025-10' AND '2026-09'
GROUP BY customer_id
""", engine)


# 3) 투자자 기준으로 결합
investment_feature = investor_base.merge(
    investment_12m,
    on="customer_id",
    how="left"
)

# 혹시 월 데이터가 없는 경우 0 처리
investment_feature["investment_12m"] = (
    investment_feature["investment_12m"].fillna(0)
)


# 4) 투자 적극성
investment_feature["investment_disposable_income_ratio"] = (
    investment_feature["investment_12m"]
    / investment_feature["disposable_annual_income"]
)


# ============================================================
# 5. 기본 QA
# ============================================================

print("=" * 65)
print("INVESTMENT INTENSITY BASIC QA")
print("=" * 65)

print("투자자 수:", len(investment_feature))
print("고객 ID 중복:", investment_feature["customer_id"].duplicated().sum())

print("\n[결측치]")
print(
    investment_feature[
        [
            "disposable_annual_income",
            "investment_12m",
            "investment_disposable_income_ratio"
        ]
    ].isna().sum()
)

print("\n[가처분소득 <= 0 고객]")
print(
    (investment_feature["disposable_annual_income"] <= 0).sum()
)

print("\n[투자금액 < 0 고객]")
print(
    (investment_feature["investment_12m"] < 0).sum()
)


# ============================================================
# 6. 투자 적극성 분포
# ============================================================

print("\n" + "=" * 65)
print("INVESTMENT INTENSITY DISTRIBUTION")
print("=" * 65)

display(
    investment_feature[
        "investment_disposable_income_ratio"
    ].describe(
        percentiles=[.01, .05, .10, .25, .33, .50, .67, .75, .90, .95, .99]
    ).to_frame()
)

INVESTMENT INTENSITY BASIC QA
투자자 수: 4079
고객 ID 중복: 0

[결측치]
disposable_annual_income              0
investment_12m                        0
investment_disposable_income_ratio    0
dtype: int64

[가처분소득 <= 0 고객]
0

[투자금액 < 0 고객]
0

INVESTMENT INTENSITY DISTRIBUTION


,investment_disposable_income_ratio
count,4079.000000
mean,0.050279
std,0.017758
min,0.000000
1%,0.000000
5%,0.000000
10%,0.030405
25%,0.043480
33%,0.046750
50%,0.053450


In [7]:
# ============================================================
# 7. 투자 적극성 3분류
# ============================================================

ratio = investment_feature["investment_disposable_income_ratio"]

# 새 DB 투자자 기준 tertile
q33 = ratio.quantile(0.33)
q67 = ratio.quantile(0.67)

print("33% 기준:", q33)
print("67% 기준:", q67)


def classify_investment_intensity(x):
    if x <= q33:
        return "소극형"
    elif x <= q67:
        return "보통형"
    else:
        return "적극형"


investment_feature["investment_intensity_type"] = (
    ratio.apply(classify_investment_intensity)
)


# ============================================================
# 8. 분류 결과 확인
# ============================================================

intensity_summary = (
    investment_feature
    .groupby("investment_intensity_type")
    .agg(
        customer_count=("customer_id", "count"),
        mean_ratio=("investment_disposable_income_ratio", "mean"),
        median_ratio=("investment_disposable_income_ratio", "median"),
        min_ratio=("investment_disposable_income_ratio", "min"),
        max_ratio=("investment_disposable_income_ratio", "max")
    )
)

intensity_summary["percentage"] = (
    intensity_summary["customer_count"]
    / len(investment_feature)
    * 100
)

# 보기 좋은 순서
intensity_summary = intensity_summary.reindex(
    ["소극형", "보통형", "적극형"]
)

print("=" * 65)
print("INVESTMENT INTENSITY CLASSIFICATION")
print("=" * 65)

display(intensity_summary)

print("\n분류 결측치:",
      investment_feature["investment_intensity_type"].isna().sum())

print("\n총 인원:",
      intensity_summary["customer_count"].sum())

33% 기준: 0.04675000065667182
67% 기준: 0.059299998969707265
INVESTMENT INTENSITY CLASSIFICATION


,customer_count,mean_ratio,median_ratio,min_ratio,max_ratio,percentage
investment_intensity_type,,,,,,
소극형,1346,0.030999,0.038345,0.00000,0.04675,32.998284
보통형,1387,0.053298,0.053450,0.04675,0.05930,34.003432
적극형,1346,0.066448,0.065095,0.05930,0.09291,32.998284



분류 결측치: 0

총 인원: 4079


In [8]:
# ============================================================
# 9. 포트폴리오 HHI 계산
# ============================================================

# 주식 보유 데이터 불러오기
stock_positions = pd.read_sql("""
SELECT
    customer_id,
    stock_code,
    quantity,
    purchase_price_actual_close
FROM customer_stock_positions
""", engine)


print("=" * 65)
print("STOCK POSITION BASIC QA")
print("=" * 65)

print("전체 행 수:", len(stock_positions))
print("전체 고객 수:", stock_positions["customer_id"].nunique())

print("\n[결측치]")
print(
    stock_positions[
        ["customer_id", "stock_code",
         "quantity", "purchase_price_actual_close"]
    ].isna().sum()
)

print("\n[quantity <= 0]")
print((stock_positions["quantity"] <= 0).sum())

print("\n[purchase_price_actual_close <= 0]")
print((stock_positions["purchase_price_actual_close"] <= 0).sum())


# ============================================================
# 10. 종목별 매입금액 계산
# ============================================================

stock_positions["purchase_value"] = (
    stock_positions["quantity"]
    * stock_positions["purchase_price_actual_close"]
)


# ============================================================
# 11. 고객-종목 단위로 합산
# ============================================================
# 혹시 동일 고객이 동일 종목을 여러 행 보유하더라도
# 종목별 금액을 먼저 합산한 후 HHI 계산

stock_by_customer = (
    stock_positions
    .groupby(["customer_id", "stock_code"], as_index=False)
    .agg(
        purchase_value=("purchase_value", "sum")
    )
)


# ============================================================
# 12. 고객별 총 매입금액
# ============================================================

stock_by_customer["total_purchase_value"] = (
    stock_by_customer
    .groupby("customer_id")["purchase_value"]
    .transform("sum")
)


# 종목별 포트폴리오 비중
stock_by_customer["weight"] = (
    stock_by_customer["purchase_value"]
    / stock_by_customer["total_purchase_value"]
)

# 비중 제곱
stock_by_customer["weight_squared"] = (
    stock_by_customer["weight"] ** 2
)


# ============================================================
# 13. 고객별 HHI
# ============================================================

portfolio_feature = (
    stock_by_customer
    .groupby("customer_id", as_index=False)
    .agg(
        stock_count=("stock_code", "nunique"),
        total_purchase_value=("purchase_value", "sum"),
        portfolio_hhi=("weight_squared", "sum")
    )
)


# ============================================================
# 14. HHI QA
# ============================================================

print("\n" + "=" * 65)
print("PORTFOLIO HHI BASIC QA")
print("=" * 65)

print("고객 수:", len(portfolio_feature))
print(
    "고객 ID 중복:",
    portfolio_feature["customer_id"].duplicated().sum()
)

print("\n[결측치]")
print(portfolio_feature.isna().sum())

print("\n[stock_count]")
display(
    portfolio_feature["stock_count"]
    .describe()
    .to_frame()
)

print("\n[portfolio_hhi]")
display(
    portfolio_feature["portfolio_hhi"]
    .describe(
        percentiles=[
            .01, .05, .10, .25, .33,
            .50, .67, .75, .90, .95, .99
        ]
    )
    .to_frame()
)

print("\n[HHI 범위 오류]")
print(
    "HHI <= 0:",
    (portfolio_feature["portfolio_hhi"] <= 0).sum()
)

print(
    "HHI > 1:",
    (portfolio_feature["portfolio_hhi"] > 1 + 1e-10).sum()
)

print("\n[보유종목 수별 고객 수]")
display(
    portfolio_feature["stock_count"]
    .value_counts()
    .sort_index()
    .rename("customer_count")
    .to_frame()
)

STOCK POSITION BASIC QA
전체 행 수: 12832
전체 고객 수: 4079

[결측치]
customer_id                    0
stock_code                     0
quantity                       0
purchase_price_actual_close    0
dtype: int64

[quantity <= 0]
0

[purchase_price_actual_close <= 0]
0

PORTFOLIO HHI BASIC QA
고객 수: 4079
고객 ID 중복: 0

[결측치]
customer_id             0
stock_count             0
total_purchase_value    0
portfolio_hhi           0
dtype: int64

[stock_count]


,stock_count
count,4079.000000
mean,3.145869
std,1.452384
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,6.000000



[portfolio_hhi]


,portfolio_hhi
count,4079.000000
mean,0.531245
std,0.246609
min,0.170869
1%,0.204821
5%,0.241840
10%,0.268927
25%,0.340968
33%,0.369042
50%,0.469967



[HHI 범위 오류]
HHI <= 0: 0
HHI > 1: 0

[보유종목 수별 고객 수]


,customer_count
stock_count,
1,576
2,917
3,1010
4,802
5,460
6,314


In [9]:
# ============================================================
# 포트폴리오 집중도 분류
# 기존 최종 로직: 논문/ENA 기반 절대 기준
# ============================================================

def classify_portfolio_concentration(hhi):
    if hhi <= 0.3:
        return "분산형"
    elif hhi <= 0.5:
        return "중간형"
    else:
        return "집중형"


portfolio_feature["portfolio_concentration_type"] = (
    portfolio_feature["portfolio_hhi"]
    .apply(classify_portfolio_concentration)
)


# ============================================================
# 분류 결과 확인
# ============================================================

portfolio_summary = (
    portfolio_feature
    .groupby("portfolio_concentration_type")
    .agg(
        customer_count=("customer_id", "count"),
        mean_hhi=("portfolio_hhi", "mean"),
        median_hhi=("portfolio_hhi", "median"),
        min_hhi=("portfolio_hhi", "min"),
        max_hhi=("portfolio_hhi", "max"),
        mean_stock_count=("stock_count", "mean")
    )
)

portfolio_summary["percentage"] = (
    portfolio_summary["customer_count"]
    / len(portfolio_feature)
    * 100
)

portfolio_summary = portfolio_summary.reindex(
    ["분산형", "중간형", "집중형"]
)

print("=" * 65)
print("PORTFOLIO CONCENTRATION - PAPER BASED")
print("=" * 65)

display(portfolio_summary)

print(
    "\n분류 결측치:",
    portfolio_feature["portfolio_concentration_type"].isna().sum()
)

print(
    "총 투자자 수:",
    portfolio_summary["customer_count"].sum()
)

PORTFOLIO CONCENTRATION - PAPER BASED


,customer_count,mean_hhi,median_hhi,min_hhi,max_hhi,mean_stock_count,percentage
portfolio_concentration_type,,,,,,,
분산형,662,0.255890,0.260628,0.170869,0.299982,5.132931,16.229468
중간형,1505,0.385712,0.376105,0.300118,0.499696,3.752159,36.896298
집중형,1912,0.741136,0.679716,0.500000,1.000000,1.980649,46.874234



분류 결측치: 0
총 투자자 수: 4079


In [10]:
# ============================================================
# FINAL FEATURE TABLE 생성
# ============================================================

# 1. 전체 고객 기준 테이블
final_feature = pd.read_sql("""
SELECT
    customer_id,
    is_stock_investor,
    disposable_annual_income
FROM customer_financial_profiles
""", engine)


# ============================================================
# 2. 투자 적극성 Feature 결합
# ============================================================

intensity_cols = [
    "customer_id",
    "investment_12m",
    "investment_disposable_income_ratio",
    "investment_intensity_type"
]

final_feature = final_feature.merge(
    investment_feature[intensity_cols],
    on="customer_id",
    how="left"
)


# ============================================================
# 3. 포트폴리오 Feature 결합
# ============================================================

portfolio_cols = [
    "customer_id",
    "stock_count",
    "total_purchase_value",
    "portfolio_hhi",
    "portfolio_concentration_type"
]

final_feature = final_feature.merge(
    portfolio_feature[portfolio_cols],
    on="customer_id",
    how="left"
)


# ============================================================
# 4. 비투자자 유형 처리
# ============================================================

final_feature["investment_intensity_type"] = (
    final_feature["investment_intensity_type"]
    .fillna("미해당")
)

final_feature["portfolio_concentration_type"] = (
    final_feature["portfolio_concentration_type"]
    .fillna("미해당")
)


# ============================================================
# 5. 최종 컬럼 순서
# ============================================================

final_feature = final_feature[
    [
        "customer_id",
        "is_stock_investor",
        "disposable_annual_income",
        "investment_12m",
        "investment_disposable_income_ratio",
        "investment_intensity_type",
        "stock_count",
        "total_purchase_value",
        "portfolio_hhi",
        "portfolio_concentration_type"
    ]
]


print("=" * 65)
print("FINAL FEATURE TABLE CREATED")
print("=" * 65)

print("테이블 크기:", final_feature.shape)

display(final_feature.head())

FINAL FEATURE TABLE CREATED
테이블 크기: (13852, 10)


,customer_id,is_stock_investor,disposable_annual_income,investment_12m,investment_disposable_income_ratio,investment_intensity_type,stock_count,total_purchase_value,portfolio_hhi,portfolio_concentration_type
0,18,1,51368860.25,3054906.12,0.059470,적극형,2.0,4212560.0,0.623672,집중형
1,39,1,62830989.15,3554349.00,0.056570,보통형,3.0,807175.0,0.546103,집중형
2,42,0,82366539.33,NaN,NaN,미해당,NaN,NaN,NaN,미해당
3,44,1,25951381.22,1623898.60,0.062575,적극형,2.0,6170360.0,0.836634,집중형
4,47,1,88321850.07,6918250.56,0.078330,적극형,3.0,1081276.0,0.476239,중간형


In [11]:
# ============================================================
# FINAL QA : INVESTMENT FEATURE TABLE
# ============================================================

print("=" * 65)
print("FINAL QA : INVESTMENT FEATURE TABLE")
print("=" * 65)


# ------------------------------------------------------------
# 1. 기본 구조
# ------------------------------------------------------------

print("\n[1. 기본 구조]")

print("테이블 크기:", final_feature.shape)
print("전체 고객 수:", final_feature["customer_id"].nunique())
print("고객 ID 중복:", final_feature["customer_id"].duplicated().sum())

print(
    "투자자 수:",
    (final_feature["is_stock_investor"] == 1).sum()
)

print(
    "비투자자 수:",
    (final_feature["is_stock_investor"] == 0).sum()
)


# ------------------------------------------------------------
# 2. 전체 컬럼 결측치
# ------------------------------------------------------------

print("\n[2. 전체 컬럼 결측치]")

print(final_feature.isna().sum())


# ------------------------------------------------------------
# 3. 투자자 데이터 완전성
# ------------------------------------------------------------

investor_final = final_feature[
    final_feature["is_stock_investor"] == 1
].copy()

print("\n[3. 투자자 데이터 완전성]")

print(
    investor_final[
        [
            "disposable_annual_income",
            "investment_12m",
            "investment_disposable_income_ratio",
            "investment_intensity_type",
            "stock_count",
            "total_purchase_value",
            "portfolio_hhi",
            "portfolio_concentration_type"
        ]
    ].isna().sum()
)


# ------------------------------------------------------------
# 4. 비투자자 처리 확인
# ------------------------------------------------------------

non_investor_final = final_feature[
    final_feature["is_stock_investor"] == 0
].copy()

print("\n[4. 비투자자 처리 확인]")

numeric_investment_cols = [
    "investment_12m",
    "investment_disposable_income_ratio",
    "stock_count",
    "total_purchase_value",
    "portfolio_hhi"
]

print("비투자자 수:", len(non_investor_final))

print("\n비투자자 숫자형 투자변수 non-null 개수:")
print(
    non_investor_final[
        numeric_investment_cols
    ].notna().sum()
)

print(
    "\n투자 적극성 유형:",
    non_investor_final[
        "investment_intensity_type"
    ].value_counts()
)

print(
    "\n포트폴리오 집중도 유형:",
    non_investor_final[
        "portfolio_concentration_type"
    ].value_counts()
)


# ------------------------------------------------------------
# 5. 투자 적극성 분류
# ------------------------------------------------------------

print("\n[5. 투자 적극성 분류]")

print(
    investor_final[
        "investment_intensity_type"
    ].value_counts()
)


# ------------------------------------------------------------
# 6. 포트폴리오 집중도 분류
# ------------------------------------------------------------

print("\n[6. 포트폴리오 집중도 분류]")

print(
    investor_final[
        "portfolio_concentration_type"
    ].value_counts()
)


# ------------------------------------------------------------
# 7. 값 범위 QA
# ------------------------------------------------------------

print("\n[7. 값 범위 QA]")

print(
    "투자 적극성 < 0:",
    (
        investor_final[
            "investment_disposable_income_ratio"
        ] < 0
    ).sum()
)

print(
    "HHI <= 0:",
    (investor_final["portfolio_hhi"] <= 0).sum()
)

print(
    "HHI > 1:",
    (investor_final["portfolio_hhi"] > 1 + 1e-10).sum()
)


# ------------------------------------------------------------
# 8. 최종 샘플
# ------------------------------------------------------------

print("\n[8. 투자자 샘플]")

display(
    investor_final.head(10)
)

FINAL QA : INVESTMENT FEATURE TABLE

[1. 기본 구조]
테이블 크기: (13852, 10)
전체 고객 수: 13852
고객 ID 중복: 0
투자자 수: 4079
비투자자 수: 9773

[2. 전체 컬럼 결측치]
customer_id                              0
is_stock_investor                        0
disposable_annual_income                 0
investment_12m                        9773
investment_disposable_income_ratio    9773
investment_intensity_type                0
stock_count                           9773
total_purchase_value                  9773
portfolio_hhi                         9773
portfolio_concentration_type             0
dtype: int64

[3. 투자자 데이터 완전성]
disposable_annual_income              0
investment_12m                        0
investment_disposable_income_ratio    0
investment_intensity_type             0
stock_count                           0
total_purchase_value                  0
portfolio_hhi                         0
portfolio_concentration_type          0
dtype: int64

[4. 비투자자 처리 확인]
비투자자 수: 9773

비투자자 숫자형 투자변수 non-null 개수:
investment_1

,customer_id,is_stock_investor,disposable_annual_income,investment_12m,investment_disposable_income_ratio,investment_intensity_type,stock_count,total_purchase_value,portfolio_hhi,portfolio_concentration_type
0,18,1,5.136886e+07,3054906.12,0.059470,적극형,2.0,4212560.0,0.623672,집중형
1,39,1,6.283099e+07,3554349.00,0.056570,보통형,3.0,807175.0,0.546103,집중형
3,44,1,2.595138e+07,1623898.60,0.062575,적극형,2.0,6170360.0,0.836634,집중형
4,47,1,8.832185e+07,6918250.56,0.078330,적극형,3.0,1081276.0,0.476239,중간형
15,161,1,6.100817e+07,3500648.76,0.057380,보통형,1.0,10578800.0,1.000000,집중형
19,196,1,2.571386e+07,1304978.28,0.050750,보통형,1.0,1712050.0,1.000000,집중형
21,208,1,4.615710e+07,2910268.92,0.063051,적극형,1.0,4640700.0,1.000000,집중형
22,213,1,1.376914e+08,8987115.24,0.065270,적극형,5.0,3599763.0,0.257419,분산형
25,221,1,5.546018e+07,3167885.40,0.057120,보통형,5.0,33416930.0,0.243992,분산형
31,282,1,2.503346e+07,0.00,0.000000,소극형,4.0,5220570.0,0.290484,분산형


| 항목 | 최종 기준 |
|---|---|
| DB | `fintech_cohort_25_39` |
| 전체 고객 | **13,852명** |
| 투자자 | **4,079명** |
| 비투자자 | **9,773명** |
| 투자자 정의 | `is_stock_investor = 1` |
| 분석 대상 연령 | **25~39세** |
| 투자 적극성 산출 기간 | **2025-10 ~ 2026-09 (최근 12개월)** |
| 투자 적극성 | `investment_12m / disposable_annual_income` |
| `investment_12m` | 최근 12개월 `investment_transfer` 합계 |
| 적극성 분류 | 투자자 기준 **33%, 67% tertile** |
| 적극성 경계 | **2025-10~2026-09 데이터로 재산출한 q33 / q67 적용** |
| HHI | 종목별 **매입금액 비중의 제곱합** |
| 종목별 매입금액 | `quantity × purchase_price_actual_close` |
| HHI 분류 | **≤ 0.3 분산형 / 0.3 초과~0.5 이하 중간형 / > 0.5 집중형** |
| 비투자자 처리 | 투자 관련 수치 `NaN`, 유형은 비투자자로 별도 표시 |
| 최종 데이터 | **13,852 × 10** |